# Decision Tree Hyperparameter Optimization

## Objective

Optimize the Decision Tree Regression model using systematic hyperparameter tuning.

## Hyperparameters Evaluated

- max_depth
- min_samples_split
- min_samples_leaf

## Optimization Method

GridSearchCV with cross-validation.

## Selection Criterion

The model configuration producing the highest cross-validation performance was selected.

## Conclusion

The selected hyperparameter combination provided the best balance between predictive accuracy and model generalization while reducing the likelihood of overfitting.

---

## Output

The optimized Decision Tree configuration was incorporated into the final price prediction model.

In [1]:
import sqlite3
import itertools
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeRegressor

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score
)
from sklearn.model_selection import KFold, cross_validate
from sklearn.model_selection import GridSearchCV


In [2]:
# Connect to SQLite database
conn = sqlite3.connect("pakwheels_dataset.db")

# Read dataset
df = pd.read_sql_query(
    "SELECT * FROM market_analysis_cars",
    conn
)

conn.close()

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (2526, 10)


,City,Title,Price_PKR,Year,Mileage_KM,Fuel_Type,Engine_CC,Transmission,Company,Model
0,Lahore,Toyota Corolla 2018 XLi Automatic for Sale,3990000,2018,92000,Petrol,1300,Automatic,Toyota,Corolla
1,Lahore,Daihatsu Mira 2022 L SA III for Sale,3890000,2022,16000,Petrol,660,Automatic,Daihatsu,Mira
2,Lahore,Toyota Corolla 2019 XLi VVTi for Sale,3750000,2019,109000,Petrol,1300,Manual,Toyota,Corolla
3,Lahore,Toyota Yaris Sedan 2021 ATIV X CVT 1.5 for Sale,4450000,2021,117855,Petrol,1500,Automatic,Toyota,Yaris
4,Lahore,Honda City 5th (GM2) Generation 2018 1.3 i-VTE...,3500000,2018,156181,Petrol,1300,Automatic,Honda,City


In [3]:
# ----------------------------------------------------------
# Select columns required for price prediction
# ----------------------------------------------------------

required_columns = [
    'Company',
    'Model',
    'City',
    'Year',
    'Mileage_KM',
    'Engine_CC',
    'Fuel_Type',
    'Transmission',
    'Price_PKR'
]

# Keep only required columns and remove missing values
df = df[required_columns].dropna()

print("Dataset Shape After Cleaning:", df.shape)

df.head()

Dataset Shape After Cleaning: (2526, 9)


,Company,Model,City,Year,Mileage_KM,Engine_CC,Fuel_Type,Transmission,Price_PKR
0,Toyota,Corolla,Lahore,2018,92000,1300,Petrol,Automatic,3990000
1,Daihatsu,Mira,Lahore,2022,16000,660,Petrol,Automatic,3890000
2,Toyota,Corolla,Lahore,2019,109000,1300,Petrol,Manual,3750000
3,Toyota,Yaris,Lahore,2021,117855,1500,Petrol,Automatic,4450000
4,Honda,City,Lahore,2018,156181,1300,Petrol,Automatic,3500000


In [4]:
# ----------------------------------------------------------
# Define Features (X) and Target (y)
# ----------------------------------------------------------

X = df.drop('Price_PKR', axis=1)
y = df['Price_PKR']

print("Number of Features:", X.shape[1])
print("Number of Samples:", len(X))

Number of Features: 8
Number of Samples: 2526


In [5]:
# ----------------------------------------------------------
# Define Feature Types
# ----------------------------------------------------------

categorical_features = [
    'Company',
    'Model',
    'City',
    'Fuel_Type',
    'Transmission'
]

numeric_features = [
    'Year',
    'Mileage_KM',
    'Engine_CC'
]

print("Categorical Features:", categorical_features)
print("Numerical Features:", numeric_features)

Categorical Features: ['Company', 'Model', 'City', 'Fuel_Type', 'Transmission']
Numerical Features: ['Year', 'Mileage_KM', 'Engine_CC']


In [6]:
# ----------------------------------------------------------
# Train-Test Split
# ----------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Samples :", len(X_train))
print("Testing Samples  :", len(X_test))

Training Samples : 2020
Testing Samples  : 506


In [7]:
# ----------------------------------------------------------
# Preprocessing Pipeline
# ----------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            ),
            categorical_features
        ),
        (
            'num',
            'passthrough',
            numeric_features
        )
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [8]:
# ----------------------------------------------------------
# Baseline Decision Tree
# ----------------------------------------------------------

baseline_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('regressor',
            DecisionTreeRegressor(
                max_depth=8,
                min_samples_leaf=10,
                random_state=42
            )
        )
    ]
)

baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_mape = mean_absolute_percentage_error(
    y_test,
    baseline_predictions
) * 100
baseline_r2 = r2_score(y_test, baseline_predictions)

print("\n----------- BASELINE MODEL -----------")
print(f"MAE   : PKR {baseline_mae:,.0f}")
print(f"MAPE  : {baseline_mape:.2f}%")
print(f"R²    : {baseline_r2:.4f}")


----------- BASELINE MODEL -----------
MAE   : PKR 862,676
MAPE  : 17.92%
R²    : 0.8287


In [9]:
# ----------------------------------------------------------
# Hyperparameter Search Space
# ----------------------------------------------------------

param_grid = {
    "regressor__max_depth": [
        4, 5, 6, 7, 8,
        9, 10, 12, 15, 20, None
    ],

    "regressor__min_samples_leaf": [
        1, 2, 3, 5,
        8, 10, 15, 20
    ],

    "regressor__min_samples_split": [
        2, 4, 5,
        8, 10,
        15, 20
    ]
}

total_models = (
    len(param_grid["regressor__max_depth"])
    * len(param_grid["regressor__min_samples_leaf"])
    * len(param_grid["regressor__min_samples_split"])
)

print(f"Total Parameter Combinations: {total_models}")

Total Parameter Combinations: 616


In [10]:
print(grid_search.best_params_)
print(grid_search.best_estimator_)
print(results_df.dtypes)
print(results_df.head(2))

NameError: name 'grid_search' is not defined

In [10]:
# ----------------------------------------------------------
# Pipeline Used for Hyperparameter Tuning
# ----------------------------------------------------------

pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "regressor",
            DecisionTreeRegressor(
                random_state=42
            )
        )
    ]
)

print("Pipeline created successfully.")

Pipeline created successfully.


In [11]:
# ----------------------------------------------------------
# Grid Search with 5-Fold Cross Validation
# ----------------------------------------------------------

from sklearn.model_selection import KFold

# Define 5-Fold Cross Validation
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=pipeline,

    param_grid=param_grid,

    scoring="neg_mean_absolute_error",

    cv=cv,

    n_jobs=-1,

    verbose=2,

    refit=True,

    return_train_score=True
)

print("Starting Grid Search...\n")

grid_search.fit(
    X_train,
    y_train
)

print("\nGrid Search Completed.")

Starting Grid Search...

Fitting 5 folds for each of 616 candidates, totalling 3080 fits

Grid Search Completed.


In [12]:
# ----------------------------------------------------------
# Best Hyperparameters
# ----------------------------------------------------------

print("Best Hyperparameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation MAE:")
print(f"PKR {-grid_search.best_score_:,.0f}")

# ----------------------------------------------------------
# Complete Grid Search Results
# ----------------------------------------------------------

results_df = pd.DataFrame(grid_search.cv_results_)

results_df["Mean CV MAE"] = -results_df["mean_test_score"]
results_df["Std CV MAE"] = results_df["std_test_score"]

results_df["Mean Train MAE"] = -results_df["mean_train_score"]

results_df = results_df[
    [
        "param_regressor__max_depth",
        "param_regressor__min_samples_leaf",
        "param_regressor__min_samples_split",
        "Mean CV MAE",
        "Std CV MAE",
        "Mean Train MAE",
        "rank_test_score"
    ]
]

results_df = results_df.sort_values(
    by="Mean CV MAE"
).reset_index(drop=True)

print("\nTop 10 Models:\n")

display(results_df.head(10))

# ----------------------------------------------------------
# Save Complete Results
# ----------------------------------------------------------

results_df.to_csv(
    "grid_search_results.csv",
    index=False
)

print("\nComplete grid search results saved as 'grid_search_results.csv'")

Best Hyperparameters:
{'regressor__max_depth': None, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2}

Best Cross-Validation MAE:
PKR 503,281

Top 10 Models:



,param_regressor__max_depth,param_regressor__min_samples_leaf,param_regressor__min_samples_split,Mean CV MAE,Std CV MAE,Mean Train MAE,rank_test_score
0,None,1,2,503281.190594,46469.917776,178.217822,1
1,20,1,2,512558.525726,58450.737417,971.728639,2
2,15,1,2,539582.120680,46305.637032,27788.687295,3
3,12,1,2,574652.576065,56879.851307,111127.657460,4
4,20,1,4,574706.459472,65069.722142,96497.264530,5
5,15,1,4,578303.561687,56861.049880,113190.048310,6
6,None,1,5,582329.869183,42058.187427,151000.040594,7
7,None,1,4,584948.350000,65343.836382,96126.237500,8
8,15,1,5,589948.801664,48840.524798,163952.898660,9
9,20,1,5,599608.647219,57796.336620,151173.254031,10



Complete grid search results saved as 'grid_search_results.csv'


In [13]:
# ----------------------------------------------------------
# Compare Top 2 Grid Search Models on the Test Set
# ----------------------------------------------------------

from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score
)

comparison_results = []

# Select the top 2 models from Grid Search
top_models = results_df.head(2)

for index, row in top_models.iterrows():

    # Extract hyperparameters
    max_depth = row["param_regressor__max_depth"]

    # Convert NaN back to None
    if pd.isna(max_depth):
        max_depth = None
    else:
        max_depth = int(max_depth)

    min_samples_leaf = int(row["param_regressor__min_samples_leaf"])
    min_samples_split = int(row["param_regressor__min_samples_split"])

    # Create Decision Tree
    tree = DecisionTreeRegressor(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split,
        random_state=42
    )

    # Create Pipeline
    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", tree)
        ]
    )

    # Train on the full training set
    model.fit(X_train, y_train)

    # Predict on the untouched test set
    predictions = model.predict(X_test)

    # Compute metrics
    mae = mean_absolute_error(y_test, predictions)
    mape = mean_absolute_percentage_error(y_test, predictions) * 100
    r2 = r2_score(y_test, predictions)

    comparison_results.append({
        "Rank": index + 1,
        "max_depth": max_depth,
        "min_samples_leaf": min_samples_leaf,
        "min_samples_split": min_samples_split,
        "Test MAE": mae,
        "Test MAPE (%)": mape,
        "Test R²": r2
    })

comparison_df = pd.DataFrame(comparison_results)

print("Comparison of Top 2 Grid Search Models\n")

display(comparison_df)

Comparison of Top 2 Grid Search Models



,Rank,max_depth,min_samples_leaf,min_samples_split,Test MAE,Test MAPE (%),Test R²
0,1,NaN,1,2,449086.952569,10.208047,0.870870
1,2,20.0,1,2,380158.100132,9.627415,0.949047


In [14]:
# ----------------------------------------------------------
# Comparing Different Cross Validation Fold Values
# ----------------------------------------------------------

from sklearn.model_selection import KFold, GridSearchCV
import time


cv_values = [5, 10, 15, 20]

cv_comparison_results = []


for folds in cv_values:

    print(f"\nRunning {folds}-Fold Cross Validation...")

    start_time = time.time()

    cv_strategy = KFold(
        n_splits=folds,
        shuffle=True,
        random_state=42
    )

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="neg_mean_absolute_error",
        cv=cv_strategy,
        n_jobs=-1,
        verbose=0,
        refit=True
    )

    grid.fit(
        X_train,
        y_train
    )

    elapsed_time = time.time() - start_time

    cv_comparison_results.append({

        "CV Folds": folds,

        "Best CV MAE": -grid.best_score_,

        "Best Parameters": grid.best_params_,

        "Time (seconds)": elapsed_time

    })


cv_comparison_df = pd.DataFrame(
    cv_comparison_results
)


cv_comparison_df = cv_comparison_df.sort_values(
    by="Best CV MAE"
)


display(cv_comparison_df)


Running 5-Fold Cross Validation...

Running 10-Fold Cross Validation...

Running 15-Fold Cross Validation...

Running 20-Fold Cross Validation...


,CV Folds,Best CV MAE,Best Parameters,Time (seconds)
2,15,413437.220299,"{'regressor__max_depth': None, 'regressor__min...",75.005025
3,20,414933.202251,"{'regressor__max_depth': 20, 'regressor__min_s...",109.662752
1,10,432973.763861,"{'regressor__max_depth': None, 'regressor__min...",50.126242
0,5,503281.190594,"{'regressor__max_depth': None, 'regressor__min...",27.532212


In [15]:
# ----------------------------------------------------------
# Evaluate Best 15-Fold CV Model on Untouched Test Set
# ----------------------------------------------------------

from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score
)

# Get the 15-fold result
best_15fold_model = None

cv_15 = KFold(
    n_splits=15,
    shuffle=True,
    random_state=42
)

grid_15 = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv_15,
    n_jobs=-1,
    verbose=0,
    refit=True,
    return_train_score=True
)

print("Training final 15-fold CV model...")

grid_15.fit(
    X_train,
    y_train
)

print("\nBest 15-Fold Parameters:")
print(grid_15.best_params_)

print(
    f"\nBest 15-Fold CV MAE: PKR {-grid_15.best_score_:,.0f}"
)


# ----------------------------------------------------------
# Test Set Evaluation
# ----------------------------------------------------------

best_15fold_model = grid_15.best_estimator_

test_predictions = best_15fold_model.predict(
    X_test
)

test_mae = mean_absolute_error(
    y_test,
    test_predictions
)

test_mape = (
    mean_absolute_percentage_error(
        y_test,
        test_predictions
    ) * 100
)

test_r2 = r2_score(
    y_test,
    test_predictions
)


print("\n----------- 15-FOLD CV MODEL TEST PERFORMANCE -----------")

print(f"Test MAE   : PKR {test_mae:,.0f}")
print(f"Test MAPE  : {test_mape:.2f}%")
print(f"Test R²    : {test_r2:.4f}")

Training final 15-fold CV model...

Best 15-Fold Parameters:
{'regressor__max_depth': None, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2}

Best 15-Fold CV MAE: PKR 413,437

----------- 15-FOLD CV MODEL TEST PERFORMANCE -----------
Test MAE   : PKR 449,087
Test MAPE  : 10.21%
Test R²    : 0.8709


In [16]:
# ----------------------------------------------------------
# Best Hyperparameters (15-Fold CV)
# ----------------------------------------------------------

print("Best Hyperparameters (15-Fold CV):")
print(grid_15.best_params_)

print("\nBest Cross-Validation MAE:")
print(f"PKR {-grid_15.best_score_:,.0f}")


# ----------------------------------------------------------
# Complete 15-Fold Grid Search Results
# ----------------------------------------------------------

results_15_df = pd.DataFrame(
    grid_15.cv_results_
)

results_15_df["Mean CV MAE"] = (
    -results_15_df["mean_test_score"]
)

results_15_df["Std CV MAE"] = (
    results_15_df["std_test_score"]
)

results_15_df["Mean Train MAE"] = (
    -results_15_df["mean_train_score"]
)


results_15_df = results_15_df[
    [
        "param_regressor__max_depth",
        "param_regressor__min_samples_leaf",
        "param_regressor__min_samples_split",
        "Mean CV MAE",
        "Std CV MAE",
        "Mean Train MAE",
        "rank_test_score"
    ]
]


results_15_df = results_15_df.sort_values(
    by="Mean CV MAE"
).reset_index(drop=True)


print("\nTop 10 Models (15-Fold CV):\n")

display(
    results_15_df.head(10)
)

Best Hyperparameters (15-Fold CV):
{'regressor__max_depth': None, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2}

Best Cross-Validation MAE:
PKR 413,437

Top 10 Models (15-Fold CV):



,param_regressor__max_depth,param_regressor__min_samples_leaf,param_regressor__min_samples_split,Mean CV MAE,Std CV MAE,Mean Train MAE,rank_test_score
0,None,1,2,413437.220299,114691.249344,220.644650,1
1,15,1,2,429082.253203,83845.998573,28478.750662,2
2,20,1,2,430127.296151,105204.997077,1147.019221,3
3,12,1,2,472871.549743,82211.446513,114528.160010,4
4,None,1,4,486546.206405,103865.722605,83799.379799,5
5,15,1,4,491472.213444,110954.314290,102654.528151,6
6,None,1,5,504432.219199,101895.035847,135786.353580,7
7,20,1,4,506410.158247,100272.783096,84175.898579,8
8,20,1,5,510910.519156,97570.925105,135980.018381,9
9,15,1,5,519634.110986,124827.136209,149952.585611,10


In [17]:
# ----------------------------------------------------------
# Final Evaluation of Top 2 Models (15-Fold CV)
# ----------------------------------------------------------

comparison_results_15 = []

top_models_15 = results_15_df.head(2)


for index, row in top_models_15.iterrows():

    tree = DecisionTreeRegressor(
        max_depth=row["param_regressor__max_depth"],
        min_samples_leaf=row["param_regressor__min_samples_leaf"],
        min_samples_split=row["param_regressor__min_samples_split"],
        random_state=42
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", tree)
        ]
    )

    # Train on complete training data
    model.fit(
        X_train,
        y_train
    )

    # Predict on untouched test set
    predictions = model.predict(
        X_test
    )

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    mape = (
        mean_absolute_percentage_error(
            y_test,
            predictions
        ) * 100
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    comparison_results_15.append({

        "Model": f"Model {chr(65+index)}",

        "max_depth": row["param_regressor__max_depth"],

        "min_samples_leaf": row["param_regressor__min_samples_leaf"],

        "min_samples_split": row["param_regressor__min_samples_split"],

        "Test MAE": mae,

        "Test MAPE (%)": mape,

        "Test R²": r2
    })


comparison_15_df = pd.DataFrame(
    comparison_results_15
)

display(comparison_15_df)

,Model,max_depth,min_samples_leaf,min_samples_split,Test MAE,Test MAPE (%),Test R²
0,Model A,NaN,1,2,449086.952569,10.208047,0.870870
1,Model B,15.0,1,2,402621.074655,9.920314,0.934716


In [18]:
# ----------------------------------------------------------
# Final Comparison of Selected Decision Tree Candidates
# ----------------------------------------------------------

final_candidates = [
    {
        "Model": "Depth None",
        "max_depth": None,
        "min_samples_leaf": 1,
        "min_samples_split": 2
    },
    {
        "Model": "Depth 20",
        "max_depth": 20,
        "min_samples_leaf": 1,
        "min_samples_split": 2
    },
    {
        "Model": "Depth 15",
        "max_depth": 15,
        "min_samples_leaf": 1,
        "min_samples_split": 2
    }
]


final_comparison = []


for candidate in final_candidates:

    tree = DecisionTreeRegressor(
        max_depth=candidate["max_depth"],
        min_samples_leaf=candidate["min_samples_leaf"],
        min_samples_split=candidate["min_samples_split"],
        random_state=42
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", tree)
        ]
    )


    # Train on complete training set
    model.fit(
        X_train,
        y_train
    )


    # Predictions
    train_predictions = model.predict(X_train)

    test_predictions = model.predict(X_test)


    # Metrics

    train_mae = mean_absolute_error(
        y_train,
        train_predictions
    )

    test_mae = mean_absolute_error(
        y_test,
        test_predictions
    )

    test_mape = (
        mean_absolute_percentage_error(
            y_test,
            test_predictions
        ) * 100
    )

    test_r2 = r2_score(
        y_test,
        test_predictions
    )


    final_comparison.append({

        "Model": candidate["Model"],

        "max_depth": candidate["max_depth"],

        "Train MAE": train_mae,

        "Test MAE": test_mae,

        "Test MAPE (%)": test_mape,

        "Test R²": test_r2
    })


final_comparison_df = pd.DataFrame(
    final_comparison
)


final_comparison_df = final_comparison_df.sort_values(
    by="Test MAE"
).reset_index(drop=True)


display(final_comparison_df)

,Model,max_depth,Train MAE,Test MAE,Test MAPE (%),Test R²
0,Depth 20,20.0,1192.739934,380158.100132,9.627415,0.949047
1,Depth 15,15.0,25580.465039,402621.074655,9.920314,0.934716
2,Depth None,NaN,237.623762,449086.952569,10.208047,0.870870
